# 03 — Feature Engineering
Time features, lags, targets.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path().resolve().parents[0]
df = pd.read_parquet(ROOT / 'data/processed/aqi_clean.parquet')
df = df.sort_values(['City', 'Start'])

In [7]:
# time features
df['hour'] = df['Start'].dt.hour
df['dayofweek'] = df['Start'].dt.dayofweek
df['month'] = df['Start'].dt.month
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

In [8]:
# lag features for PM2.5
df['PM25_lag1'] = df.groupby('City')['PM2.5'].shift(1)
df['PM25_lag24'] = df.groupby('City')['PM2.5'].shift(24)

# rolling mean
df['PM25_roll24'] = df.groupby('City')['PM2.5'].rolling(24, min_periods=1).mean().reset_index(level=0, drop=True)

# target: PM2.5 value 24 hours into the future
df['target_PM25_24h'] = df.groupby('City')['PM2.5'].shift(-24)

# drop NA rows resulting from shifts
df = df.dropna()
print("Features created. Shape:", df.shape)

Features created. Shape: (34841, 19)


In [9]:
df.to_parquet(ROOT / 'data/processed/features.parquet')
print("Saved features.parquet")

Saved features.parquet
